# Next-token predictions

Each new token, and the other tokens the model almost picked. This uses the Azure deployment `gpt-4o-mini`, which returns logprobs.


## Client

In [5]:
import math
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
deployment = "gpt-4o-mini"


## Read the stream

In [6]:
def next_tokens(prompt, max_tokens=20):
    stream = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0,
        logprobs=True,
        top_logprobs=3,
        stream=True,
        timeout=120.0,
    )
    rows = []
    for chunk in stream:
        # Azure sends chunks with an empty choices list (prompt filter, usage).
        if not chunk.choices:
            continue
        choice = chunk.choices[0]
        piece = choice.delta.content
        if not piece or not choice.logprobs or not choice.logprobs.content:
            continue
        top = choice.logprobs.content[0].top_logprobs
        chosen = next(item for item in top if item.token == piece)
        others = [item for item in top if item.token != piece]
        rows.append(
            {
                "token": piece,
                "probability": round(math.exp(chosen.logprob), 3),
                "alternatives": [
                    f"{item.token} ({math.exp(item.logprob):.3f})" for item in others
                ],
            }
        )
    return rows


In [7]:
prompt = "In one sentence, describe the color orange to someone who has never been able to see."

for row in next_tokens(prompt):
    alts = ", ".join(row["alternatives"]) or "none"
    print(f"{row['token']!r:20} {row['probability']:.3f}   also: {alts}")


'Orange'             0.998   also: Imagine (0.002), The (0.000)
' is'                0.992   also:  feels (0.008),  can (0.000)
' the'               0.903   also:  like (0.058),  a (0.040)
' warm'              0.574   also:  vibrant (0.211),  warmth (0.211)
','                  0.620   also:  and (0.293),  sensation (0.074)
' vibrant'           0.848   also:  invigor (0.115),  energ (0.011)
' sensation'         0.516   also:  feeling (0.456),  energy (0.011)
' of'                0.813   also:  you (0.160),  that (0.025)
' sunlight'          0.457   also:  a (0.190),  the (0.190)
' on'                0.970   also:  mixed (0.007),  warming (0.004)
' your'              1.000   also:  a (0.000),  skin (0.000)
' skin'              0.999   also:  face (0.001), skin (0.000)
' combined'          0.393   also:  mixed (0.306),  during (0.211)
' with'              1.000   also:  the (0.000), with (0.000)
' the'               1.000   also:  a (0.000),  sweet (0.000)
' sweet'             0.240   al